# Notebook 9 — Hoare Logic: Partial Correctness

**Covers Chapter 4 §4.1–4.5.** We move from "how a program runs" (operational semantics, chapters 1–2) and "how slow it is" (complexity, chapter 3) to **"can we *prove* it does the right thing"** (correctness).

The tool is **Hoare logic** — a system of inference rules for deriving statements like:

$$\{P\}\ S\ \{Q\}$$

Read as: **"if precondition `P` holds before running statement `S`, and `S` terminates, then postcondition `Q` holds after."**

## What you'll be able to do by the end

- State a Hoare triple precisely, with both conditions and the program in between.
- Apply the 6 partial-correctness rules to derive triples from primitive facts.
- Find a loop invariant for a small program and prove it really is one.
- Use the rule of consequence to glue derivations together when conditions don't quite match.
- Solve exercises 30 and 31.

## Active mode

Hoare-logic reasoning is mostly on paper, not in code. But the interpreter helps: the new `verify_triple()` and `report_triple()` functions empirically check a triple by running the program on a sample of states — useful for catching obviously wrong pre/post conditions before you commit to a full proof.

In [ ]:
import sys
from pathlib import Path
for candidate in [Path.cwd(), Path.cwd() / 'notebooks', Path.cwd().parent / 'notebooks']:
    if (candidate / 'while_lang.py').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise ImportError('could not find while_lang.py')

from while_lang import (
    parse, run, trace,
    A, B, step,
    count_steps, verify_triple, report_triple,    # NEW in chapter 4
    StepBudgetExceeded,
)
print('imports OK')

## §1. What does "correct" even mean?

From §4.1, several meanings of "correctness," stacked from cheap-to-check to expensive-to-prove:

| Property | What it means | Who checks |
|:--|:--|:--|
| **Syntactically well-formed** | Parses according to the grammar | Compiler / parser |
| **Well-typed** | Types match where required (e.g. arithmetic vs boolean) | Type checker (in typed languages) |
| **Well-behaved at runtime** | No exceptions, divisions by 0, null derefs, etc. | Runtime / static analysis |
| **Semantically correct** | Does the *right job* on every valid input | Hoare logic, model checking, formal proof |
| **Terminating** | Doesn't run forever | Termination proofs (loop variants) |

While programs are auto-passed for the first three (the grammar rejects bad syntax; there's no type system to fail; there's no runtime exception model). **The interesting question is the fourth: "does it compute the right answer?"** That's what Hoare logic addresses.

Termination is treated separately. **Partial correctness** ignores it ("if it terminates, the answer is right"). **Total correctness** demands both ("it terminates AND the answer is right"). We do partial first; total in N10.

## §2. Predicates as conditions on states

A **predicate** is a property of a state — given a state σ, it's either true or false. Mathematically: `St → 𝔹`.

We write predicates as logical/arithmetic formulas over variable names, e.g.:

$$P : \quad m \ge 0 \;\land\; n > 0$$

Reading: "the predicate that holds in any state where `m` is non-negative and `n` is positive."

**The predicate language is much richer than the While language.** We can use:

- Integer operations *not* in While: `m div n`, `m mod n`, `√n`, `m^n`, factorials, …
- Quantifiers: `∀i ∈ ℕ`, `∃k ∈ ℤ`.
- Arbitrary mathematical facts: prime factorisation, divisibility, anything from number theory.
- All four logical connectives: `∧`, `∨`, `¬`, `→`.

Why so generous? Because **we're not running predicates** — we're using them to *describe* what should be true. They live in the meta-language, not in the program.

### Predicates with substitution: `P[x ↦ a]`

The notation `P[x ↦ a]` means "the predicate `P` with `x` replaced by `a`." If `P` is `x > 5`, then `P[x ↦ x + 1]` is `x + 1 > 5` — equivalently `x > 4`.

**Why we need this:** the assignment rule (§3) is naturally written backwards — given a postcondition, we substitute to get the precondition. More on that in a moment.

## §3. Hoare triples — `{P} S {Q}`

A **Hoare triple** is the assertion:

$$\{P\}\ S\ \{Q\}$$

**Read it as:** "If S is started in a state satisfying P, **and S terminates**, then it terminates in a state satisfying Q."

Three pieces:

- **`P`** — the *precondition*. The set of input states we care about.
- **`S`** — the While statement.
- **`Q`** — the *postcondition*. What we promise about the output state.

**This is partial correctness.** If `S` doesn't terminate, the triple is *vacuously* satisfied — we said nothing about the no-termination case.

### Why predicates are richer than While booleans

While's boolean expressions can only test things you can express in `=`, `≤`, `∧`, `¬`. But we want to *say* things like "d is the integer quotient of m by n" — which means `m = d·n + r ∧ 0 ≤ r < n`. **The predicate language has `<`, `·`, `mod`, `div`, ∃, ∀** — none of which are in While. They live one level up, in the proof system.

### Ghost variables — referring to original input values

The chapter's first attempt at the division spec was:

$$P : \quad m \ge 0 \,\land\, n > 0 \qquad Q : \quad m = d \cdot n + r$$

**This is broken.** Counter-example: `m := 0; n := 1; d := 0; r := 0` satisfies the triple trivially but doesn't compute division of any specific input. The variables `m` and `n` got reset.

**The fix:** introduce *ghost variables* — italic mathematical names that capture the *original* input values and never change:

$$P : \quad m = \mathit{m} \,\land\, \mathit{m} \ge 0 \,\land\, n = \mathit{n} \,\land\, \mathit{n} > 0$$
$$Q : \quad \mathit{m} = d \cdot \mathit{n} + r \,\land\, 0 \le r < \mathit{n}$$

Ghost variables are *logical entities* — they don't appear in the program. They exist only in the predicates, to remember what the inputs *were* even after the program mutates them.

**Always check your spec for the trivial-program attack.** If `m := 0; ...` makes the postcondition true, you're missing a ghost variable somewhere.

In [ ]:
# Demonstrate the trivial-program attack on the broken spec.
trivial = 'm := 0; n := 1; d := 0; r := 0'

# Broken spec: just m = d·n + r, no ghost variables.
report_triple(
    precond=lambda s: s.get('m', 0) >= 0 and s.get('n', 0) > 0,
    prog=trivial,
    postcond=lambda s: s.get('m', 0) == s.get('d', 0) * s.get('n', 0) + s.get('r', 0),
    sample_states=[{'m': m, 'n': n} for m in range(5) for n in range(1, 5)],
    mode='partial',
    label='trivial-program attack on broken spec',
)
# It 'verifies' — but the program does nothing useful! The fix is ghost variables
# that pin the original m, n values into the post-condition.

## §4. The six rules of partial correctness

From §4.4. Six rules. Two base cases, four step cases.

### Rule 1 — Assignment (`:=`)

$$\{P[x \mapsto \mathcal{A}\,a]\}\ x := a\ \{P\}$$

**Reads:** if the postcondition `P` would hold after substituting `a` for `x`, then assigning `a` to `x` makes `P` hold.

**The trick: read the rule backwards.** Given a desired post `P`, you compute the necessary pre by substituting. Examples:

| Want post | Statement | Then pre is |
|:--|:--|:--|
| `x = 5` | `x := 5` | `5 = 5` (i.e. true) |
| `x > 0` | `x := x + 1` | `x + 1 > 0`, i.e. `x ≥ 0` |
| `x = 2y` | `x := 2 * y` | `2y = 2y` (i.e. true) |
| `x + y = 10` | `x := x − 1` | `(x − 1) + y = 10`, i.e. `x + y = 11` |

### Rule 2 — Skip

$$\{P\}\ \textbf{skip}\ \{P\}$$

Anything true before is still true after. Trivial.

### Rule 3 — Composition (`;`)

$$\dfrac{\{P\}\ S\ \{Q\} \qquad \{Q\}\ S'\ \{R\}}{\{P\}\ S;S'\ \{R\}}$$

Glue two triples together via a shared middle-condition `Q`. The output of S becomes the input of S'.

### Rule 4 — Conditional (`if`)

$$\dfrac{\{P \,\land\, \mathcal{B}\,b\}\ S\ \{Q\} \qquad \{P \,\land\, \neg\,\mathcal{B}\,b\}\ S'\ \{Q\}}{\{P\}\ \textbf{if } b \textbf{ then } S \textbf{ else }(S')\ \{Q\}}$$

**Both branches must establish the same `Q`** — otherwise we couldn't say anything definite about the post-state. The branches' preconditions inherit `P` plus information about whether `b` was true or false.

### Rule 5 — While (loop invariant)

$$\dfrac{\{P \,\land\, \mathcal{B}\,b\}\ S\ \{P\}}{\{P\}\ \textbf{while } b \textbf{ do } S\ \{P \,\land\, \neg\,\mathcal{B}\,b\}}$$

**The most important rule.** `P` is the **loop invariant** — a predicate that:

1. Holds before the loop starts.
2. Is *preserved* by the body whenever `b` is true.
3. Therefore holds after the loop, plus `¬b` (because that's why we exited).

**Finding loop invariants is an art.** Heuristics in §8.

### Rule 6 — Consequence

$$\dfrac{P \to P' \quad \{P'\}\ S\ \{Q'\} \quad Q' \to Q}{\{P\}\ S\ \{Q\}}$$

**Strengthen the precondition, weaken the postcondition.** 

Reading: if you have a triple, and your actual precondition is *stronger* than the one in the triple (it implies it), and your desired postcondition is *weaker* than what the triple guarantees, you can plug them in.

**Use case:** the composition rule needs `Q` (post of S) to literally equal `Q` (pre of S'). When they don't match exactly but the first implies the second, consequence saves you.

### Worked examples — assignment rule chained

From §4.4.4, the assignment rule "behaves as one might expect." Some quick exercises in reading it backwards:

**Example 1.** What's the strongest precondition we can put before `x := 1` to guarantee `x = 1` afterwards?

By Rule 1: `{(x = 1)[x ↦ 1]} x := 1 {x = 1}`. The substitution `(x = 1)[x ↦ 1]` replaces `x` with `1`, giving `1 = 1` — which is true (call it `tt`).

So `{tt} x := 1 {x = 1}` — *no* precondition needed.

**Example 2.** What pre guarantees `x > 6` after `x := x + 1`?

Substitute: `(x > 6)[x ↦ x + 1]` gives `x + 1 > 6`, i.e. `x > 5`. So `{x > 5} x := x + 1 {x > 6}`.

**Example 3.** What pre guarantees `x ∈ ℕ` after `y := y + 1` (where `y` and `x` are different variables)?

The assignment doesn't touch `x`, so `(x ∈ ℕ)[y ↦ y + 1] = x ∈ ℕ`. So `{x ∈ ℕ} y := y + 1 {x ∈ ℕ}`. 

**Variables not touched are preserved.** This is just the substitution doing nothing when `x ≠ y`.

## §5. The full division proof (§4.5)

Putting it all together. The program:

```
r := m;
d := 0;
while n ≤ r do (d := d + 1; r := r − n)
```

We want to prove:

$$\{P\}\ S\ \{Q\}$$

where

- **`P` :** `m = m̄ ∧ m̄ ≥ 0 ∧ n = n̄ ∧ n̄ > 0`  (capture inputs as ghosts `m̄`, `n̄`).
- **`Q` :** `m̄ = d · n̄ + r ∧ 0 ≤ r < n̄`.

### Step 1 — initial assignments

By the `:=` rule (twice) and `;`:

$$\{P\}\ r := m;\ d := 0\ \{P \,\land\, r = m̄ \,\land\, d = 0\}$$

Reading: after the assignments, we still know everything from `P`, plus the new facts that `r` got the value of `m̄` and `d` got `0`.

We can strengthen the post by noting it implies `m̄ = 0 · n̄ + m̄ = d·n̄ + r ∧ 0 ≤ r`. So:

$$\{P\}\ r := m;\ d := 0\ \{m̄ = d \cdot n̄ + r \,\land\, 0 \le r\}$$

### Step 2 — the loop invariant

Pick **`I` = `m̄ = d · n̄ + r ∧ 0 ≤ r`** as the loop invariant. We need to verify the body preserves it.

**Body:** `d := d + 1; r := r − n`. Need to show:

$$\{I \,\land\, n ≤ r\}\ d := d + 1;\ r := r − n\ \{I\}$$

Apply `:=` rule backwards on the post `m̄ = d · n̄ + r ∧ 0 ≤ r`:

- After `r := r − n`: substitute `r` with `r − n`. Get `m̄ = d · n̄ + (r − n) ∧ 0 ≤ r − n`.
- After `d := d + 1` (substituting `d` with `d + 1`): `m̄ = (d + 1) · n̄ + (r − n) ∧ 0 ≤ r − n`.

So we need the precondition `m̄ = (d + 1) · n̄ + (r − n) ∧ 0 ≤ r − n` to hold given `I ∧ n ≤ r`.

**Algebra:**
- `(d + 1) · n̄ + (r − n) = d · n̄ + n̄ + r − n` — and since `n = n̄`, this simplifies to `d · n̄ + r`.
- `I` already gives `m̄ = d · n̄ + r`. ✓
- `n ≤ r` gives `0 ≤ r − n`. ✓

**Body preserves `I`.** ✓

### Step 3 — apply the while rule

$$\{I\}\ \textbf{while } n \le r \textbf{ do } (\dots)\ \{I \,\land\, \neg(n \le r)\}$$

After the loop: `m̄ = d · n̄ + r ∧ 0 ≤ r ∧ ¬(n ≤ r)` — equivalently `m̄ = d · n̄ + r ∧ 0 ≤ r < n̄`.

**That's exactly `Q`.** ✓

### Step 4 — compose with the initial assignments via consequence

The post of step 1 (`m̄ = d · n̄ + r ∧ 0 ≤ r`) is exactly `I` — the precondition for step 3. By the `;` rule:

$$\{P\}\ S\ \{Q\}$$

**Done.** ∎

### The compact derivation format

Writing the full proof tree is cumbersome, so the chapter (and the exam) use a numbered-line format. Here's the division proof in that style:

```
1. {P} r := m; d := 0 {P ∧ r = m̄ ∧ d = 0 ∧ m̄ = d·n̄ + r ∧ 0 ≤ r}    by := and ;
2. {m̄ = d·n̄ + r ∧ 0 ≤ r ∧ n ≤ r} d := d + 1; r := r − n
    {m̄ = d·n̄ + r ∧ 0 ≤ r}                                          by := and ;
3. {m̄ = d·n̄ + r ∧ 0 ≤ r} while n ≤ r do (...)
    {m̄ = d·n̄ + r ∧ 0 ≤ r < n̄}                                       from 2, while rule
4. {P} S {m̄ = d·n̄ + r ∧ 0 ≤ r < n̄}                                  from 1, 3, ; and consequence
```

Each line cites the rule used and what previous lines it depends on. **This is the format you'll write on the exam.**

In [ ]:
# Empirical sanity check on the division proof — should produce zero counter-examples.
div = '''
    r := m;
    d := 0;
    while n <= r do (
        d := d + 1;
        r := r - n
    )
'''

# Sample states and check post: for each, capture the ORIGINAL m, n via closure.
ok_count = 0
for m_in in range(0, 20):
    for n_in in range(1, 8):
        # Pre: m, n match the captured values; m ≥ 0; n > 0.  All satisfied by construction.
        # Post: m_in = d*n_in + r AND 0 ≤ r < n_in.
        result = verify_triple(
            precond  = lambda s, m=m_in, n=n_in: s.get('m', 0) == m and s.get('n', 0) == n,
            prog     = div,
            postcond = lambda s, m=m_in, n=n_in: m == s.get('d', 0) * n + s.get('r', 0) and 0 <= s.get('r', 0) < n,
            sample_states=[{'m': m_in, 'n': n_in}],
            mode='partial',
        )
        ok_count += result['verified']
        if result['failed']:
            print(f'FAIL on m={m_in}, n={n_in}: {result["failed"]}')

print(f'\n{ok_count} cases verified (no counter-examples)')

## §6. Exercises

## Exercise 30 — why is partial correctness still useful?

> Knowing that `{P} S {⇓ Q}` holds is stronger than its counterpart `{P} S {Q}`. Justify that `{P} S {Q}` allows us to assert properties that are useful when it comes to describing the behaviour of While programs.

### Answer

Total correctness is strictly stronger because it bundles termination with the postcondition. So why bother with the partial version?

**1. Many useful programs are *not* expected to terminate on all inputs.**

The Diophantine search from Exercise 7 (find `x, y ∈ ℕ` with `mx + ny = k`) loops forever when no solution exists. We *cannot* prove total correctness — there's no postcondition we can guarantee for non-existent inputs. But we *can* prove partial correctness: "if it terminates, the values it found really are a solution." That's still a useful guarantee.

**2. Partial correctness composes cleanly.**

The composition rule `{P} S {Q}, {Q} S' {R} ⟹ {P} S;S' {R}` works for partial correctness without any extra termination hypothesis. Total correctness needs `S` to terminate before `S'` can even start.

**3. Termination is a *separable* concern.**

Often we can prove the *correctness logic* once, and then prove termination separately for the inputs we care about. The partial-correctness proof reuses with minor tweaks (mostly adding a loop variant) to give total correctness.

**4. For unknown-termination programs, partial correctness is the best we can hope for.**

The Collatz-conjecture program in §4.7.4 is a famous example. Whether it terminates for all natural numbers is *an open problem in mathematics*. We *can* prove partial correctness: if it does terminate, the result is `1`. We cannot currently prove total correctness. The partial result is still rigorous and meaningful.

## Exercise 31 — extending the β proof

> For the program S that implements the function β: ℤ → ℕ:
> ```
> if 0 ≤ n then x := 2 × n
> else x := −2 × n − 1
> ```
> (a) Give a Hoare triple that verifies this program implements the given function.
> (b) Adjust the derivation given in §4.7.1 to establish that triple.

### Recall: β is defined as

$$\beta(n) = \begin{cases} 2n & \text{if } n \ge 0 \\ -2n - 1 & \text{if } n < 0 \end{cases}$$

It maps `ℤ` bijectively to `ℕ` (zigzag encoding: 0 ↦ 0, −1 ↦ 1, 1 ↦ 2, −2 ↦ 3, 2 ↦ 4, …).

### (a) The Hoare triple

We want to capture *exactly* what β computes — not just `x ∈ ℕ` (which §4.7.1 already shows), but `x = β(n̄)` where `n̄` is the original input.

$$\boxed{\{n = \bar n\}\ S\ \{\Downarrow\, x = \beta(\bar n)\}}$$

where `β(n̄) = 2·n̄ if n̄ ≥ 0 else −2·n̄ − 1`. The total-correctness arrow `⇓` is fine here because S has no loops — it always terminates.

### (b) Adjusted derivation

Mostly identical to §4.7.1, but with the more precise postcondition.

```
1. {n = n̄ ∧ 0 ≤ n} x := 2 × n {⇓ x = 2·n̄ ∧ 0 ≤ n̄}    by := rule.
                                                       Substitution: (x = 2·n̄)[x ↦ 2·n] = 2·n = 2·n̄,
                                                       which holds since n = n̄.

2. {n = n̄ ∧ ¬(0 ≤ n)} x := −2 × n − 1 {⇓ x = −2·n̄ − 1 ∧ n̄ < 0}    by := rule.
                                                                    Substitution: (x = −2·n̄ − 1)[x ↦ −2·n − 1]
                                                                    = −2·n − 1 = −2·n̄ − 1, holds since n = n̄.

3. From 1: {n = n̄ ∧ 0 ≤ n} x := 2 × n {⇓ x = β(n̄)}
                  using the definition of β when n̄ ≥ 0.    by consequence on 1.

4. From 2: {n = n̄ ∧ n < 0} x := −2 × n − 1 {⇓ x = β(n̄)}
                  using the definition of β when n̄ < 0.    by consequence on 2.

5. {n = n̄} S {⇓ x = β(n̄)}    by the if rule applied to 3 and 4.
```

The derivation works the same way — we just track *the actual computed value*, not just "is in ℕ". Empirical check:

In [ ]:
beta_prog = '''
    if 0 <= n then
        x := 2 * n
    else (
        x := -2 * n - 1
    )
'''

def beta(n):
    return 2 * n if n >= 0 else -2 * n - 1

# Verify {n = n̄} S {⇓ x = β(n̄)} for many n values.
ok = 0
for n_in in range(-20, 21):
    res = verify_triple(
        precond  = lambda s, n=n_in: s.get('n', 0) == n,
        prog     = beta_prog,
        postcond = lambda s, n=n_in: s.get('x', 0) == beta(n),
        sample_states=[{'n': n_in}],
        mode='total',
    )
    ok += res['verified']

print(f'β verified on {ok} integer inputs in [-20, 20]')

# Also show that β really is bijective onto ℕ — print a few outputs.
print()
print('Sample outputs:')
for n in [-3, -2, -1, 0, 1, 2, 3]:
    final = run(beta_prog, {'n': n})
    print(f'  β({n:>2}) = {final.get("x", 0)}')

## Summary

**Hoare triples** make precise what we mean by "a program does the right job":

$$\{P\}\ S\ \{Q\} \quad = \quad \text{if } S \text{ starts in a state satisfying } P \text{ and terminates, then it ends in a state satisfying } Q.$$

**Predicates** are richer than While booleans — they can use any maths (`mod`, `√`, `∀`, `∃`, divisibility, …) because they live in the meta-language.

**Ghost variables** (italic mathematical names) capture the original input values so the postcondition can refer to them after the program has mutated everything.

**The six rules:**

1. **`:=`** — read backwards: post `P` becomes pre `P[x ↦ a]`.
2. **`skip`** — `{P} skip {P}`.
3. **`;`** — glue via shared middle condition.
4. **`if`** — both branches must establish the same `Q`.
5. **`while`** — find a loop invariant `P` that's preserved by the body when `b` is true. The post is `P ∧ ¬b`.
6. **Consequence** — strengthen pre, weaken post.

**Finding loop invariants is hard.** It's the core skill in Hoare-logic proofs. Heuristics: involve the variables that change, relate to the desired postcondition, look at what's true after each iteration in a small example trace.

**Empirical sanity check:** `verify_triple` runs the program on a sample of states and checks `Q` holds afterwards. **Not a proof** — it can't show a triple holds in general — but it's invaluable for catching obviously wrong specs before you commit to a long proof.

**Next:** N10 covers total correctness (loop variants), four further worked examples (β, gcd, quadratic, Collatz), and exercises 32–38.